#Demo - 1

In [191]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from keras.optimizers import Adam, SGD, RMSprop
from sklearn.metrics import mean_squared_error, r2_score

from tqdm.keras import TqdmCallback
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [111]:
url = "http://users.jyu.fi/~olkhriye/ties4911/demos/demo1/Automobile_price_data_Raw_set.csv"
org_df = pd.read_csv(url)
org_df

,symboling,normalized-losses,make,fuel-type,aspiration,num-of-doors,body-style,drive-wheels,engine-location,wheel-base,...,engine-size,fuel-system,bore,stroke,compression-ratio,horsepower,peak-rpm,city-mpg,highway-mpg,price
0,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,13495.0
1,3,NaN,alfa-romero,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111.0,5000.0,21,27,16500.0
2,1,NaN,alfa-romero,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154.0,5000.0,19,26,16500.0
3,2,164.0,audi,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102.0,5500.0,24,30,13950.0
4,2,164.0,audi,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115.0,5500.0,18,22,17450.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,-1,95.0,volvo,gas,std,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,9.5,114.0,5400.0,23,28,16845.0
201,-1,95.0,volvo,gas,turbo,four,sedan,rwd,front,109.1,...,141,mpfi,3.78,3.15,8.7,160.0,5300.0,19,25,19045.0
202,-1,95.0,volvo,gas,std,four,sedan,rwd,front,109.1,...,173,mpfi,3.58,2.87,8.8,134.0,5500.0,18,23,21485.0
203,-1,95.0,volvo,diesel,turbo,four,sedan,rwd,front,109.1,...,145,idi,3.01,3.40,23.0,106.0,4800.0,26,27,22470.0


In [112]:
features = ['make', 'body-style', 'wheel-base', 'engine-size', 'horsepower', 'peak-rpm', 'highway-mpg', 'price']
df = org_df[features]
df

,make,body-style,wheel-base,engine-size,horsepower,peak-rpm,highway-mpg,price
0,alfa-romero,convertible,88.6,130,111.0,5000.0,27,13495.0
1,alfa-romero,convertible,88.6,130,111.0,5000.0,27,16500.0
2,alfa-romero,hatchback,94.5,152,154.0,5000.0,26,16500.0
3,audi,sedan,99.8,109,102.0,5500.0,30,13950.0
4,audi,sedan,99.4,136,115.0,5500.0,22,17450.0
...,...,...,...,...,...,...,...,...
200,volvo,sedan,109.1,141,114.0,5400.0,28,16845.0
201,volvo,sedan,109.1,141,160.0,5300.0,25,19045.0
202,volvo,sedan,109.1,173,134.0,5500.0,23,21485.0
203,volvo,sedan,109.1,145,106.0,4800.0,27,22470.0


In [113]:
# sns.pairplot(df)
# plt.show()

In [114]:
# for column in df.select_dtypes(include=['number']).columns:
#     sns.boxplot(x=df[column])
#     plt.title(f'Box plot of {column}')
#     plt.show()


### Checking null values

In [115]:
df.isnull().sum()

,0
make,4
body-style,1
wheel-base,0
engine-size,0
horsepower,2
peak-rpm,2
highway-mpg,0
price,4


In [116]:
df1 = df[df.isna().any(axis=1)]
df1

,make,body-style,wheel-base,engine-size,horsepower,peak-rpm,highway-mpg,price
7,NaN,wagon,105.8,136,110.0,5500.0,25,18920.0
9,audi,hatchback,99.5,131,160.0,5500.0,22,NaN
25,NaN,sedan,93.7,90,68.0,5500.0,38,6692.0
44,isuzu,sedan,94.5,90,70.0,5400.0,43,NaN
45,isuzu,sedan,94.5,90,70.0,5400.0,43,NaN
77,NaN,hatchback,93.7,92,68.0,5500.0,38,6189.0
91,nissan,NaN,94.5,97,69.0,5200.0,37,6649.0
129,porsche,hatchback,98.4,203,288.0,5750.0,28,NaN
130,renault,wagon,96.1,132,NaN,NaN,31,9295.0
131,renault,hatchback,96.1,132,NaN,NaN,31,9895.0


In [117]:
# Gonna drop the na values since there are only 13 nan values
# print(df.columns)
df = df.dropna()

### Check for outliers

In [118]:
from scipy import stats
for column in df.select_dtypes(include=['number']).columns:
    z = np.abs(stats.zscore(df[column]))
    print(f"Z-scores for {column}:")
    print(z)
    print("----------------")

Z-scores for wheel-base:
0      1.685553
1      1.685553
2      0.718106
3      0.150958
4      0.085368
         ...   
200    1.675918
201    1.675918
202    1.675918
203    1.675918
204    1.675918
Name: wheel-base, Length: 194, dtype: float64
----------------
Z-scores for engine-size:
0      0.060180
1      0.060180
2      0.585434
3      0.441199
4      0.203431
         ...   
200    0.322807
201    0.322807
202    1.086813
203    0.418308
204    0.322807
Name: engine-size, Length: 194, dtype: float64
----------------
Z-scores for horsepower:
0      0.183211
1      0.183211
2      1.328037
3      0.056404
4      0.289706
         ...   
200    0.263082
201    1.487780
202    0.795560
203    0.050091
204    0.263082
Name: horsepower, Length: 194, dtype: float64
----------------
Z-scores for peak-rpm:
0      0.233953
1      0.233953
2      0.233953
3      0.802278
4      0.802278
         ...   
200    0.595031
201    0.387785
202    0.802278
203    0.648445
204    0.595031
Name: p

In [119]:
numerical_cols = df.select_dtypes(include=['number']).columns
numerical_feature_cols = numerical_cols.drop('price')

In [120]:
# Assuming invalid range is outside of [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]
def count_outliers(column):
  Q1 = column.quantile(0.25)
  Q3 = column.quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  return len(column[(column < lower_bound) | (column > upper_bound)])

outlier_counts = {col: count_outliers(df[col]) for col in numerical_cols}
print(outlier_counts)

{'wheel-base': 3, 'engine-size': 7, 'horsepower': 5, 'peak-rpm': 2, 'highway-mpg': 3, 'price': 14}


In [121]:
df.shape

(194, 8)

In [122]:
num_fea_df = df[numerical_feature_cols]
# Rows with outliers
df[(np.abs(stats.zscore(num_fea_df)) >= 3).any(axis=1)]

,make,body-style,wheel-base,engine-size,horsepower,peak-rpm,highway-mpg,price
18,chevrolet,hatchback,88.4,61,48.0,5100.0,53,5151.0
30,honda,hatchback,86.6,92,58.0,4800.0,54,6479.0
47,jaguar,sedan,113.0,258,176.0,4750.0,19,32250.0
48,jaguar,sedan,113.0,258,176.0,4750.0,19,35550.0
49,jaguar,sedan,102.0,326,262.0,5000.0,17,36000.0
73,mercedes-benz,sedan,120.9,308,184.0,4500.0,16,40960.0
74,mercedes-benz,hardtop,112.0,304,184.0,4500.0,16,45400.0
165,toyota,sedan,94.5,98,112.0,6600.0,29,9298.0
166,toyota,hatchback,94.5,98,112.0,6600.0,29,9538.0


In [123]:
num_fea_df = df[numerical_feature_cols]
df = df[(np.abs(stats.zscore(num_fea_df)) < 3).all(axis=1)]

### FInal preprocessed output

In [124]:
df

,make,body-style,wheel-base,engine-size,horsepower,peak-rpm,highway-mpg,price
0,alfa-romero,convertible,88.6,130,111.0,5000.0,27,13495.0
1,alfa-romero,convertible,88.6,130,111.0,5000.0,27,16500.0
2,alfa-romero,hatchback,94.5,152,154.0,5000.0,26,16500.0
3,audi,sedan,99.8,109,102.0,5500.0,30,13950.0
4,audi,sedan,99.4,136,115.0,5500.0,22,17450.0
...,...,...,...,...,...,...,...,...
200,volvo,sedan,109.1,141,114.0,5400.0,28,16845.0
201,volvo,sedan,109.1,141,160.0,5300.0,25,19045.0
202,volvo,sedan,109.1,173,134.0,5500.0,23,21485.0
203,volvo,sedan,109.1,145,106.0,4800.0,27,22470.0


## Encoding & Spliting

In [128]:
# Split features and target
X = df[["make", "body-style", "wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]]
y = df["price"].astype(float)

In [129]:
# Preprocessing: One-hot encoding for categorical and scaling for numeric features
categorical_features = ["make", "body-style"]
numeric_features = ["wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(), categorical_features)
    ]
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [166]:
y_train.sort_index()

,price
0,13495.0
1,16500.0
2,16500.0
3,13950.0
4,17450.0
...,...
200,16845.0
201,19045.0
202,21485.0
203,22470.0


In [158]:
y_test.shape

(41,)

### Training the model

In [207]:
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=200,          # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value of the monitored quantity
)

# Build the DNN model
def build_model(input_dim):
    model = Sequential([
        Dense(64, activation='relu', input_dim=input_dim),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.1),
        Dense(8, activation='relu'),
        Dropout(0.1),
        Dense(1, activation='relu')  # Output layer for regression
    ])
    model.compile(optimizer=Adam(learning_rate=0.005), loss='mse', metrics=['mae'])
    return model

# Initialize the model
input_dim = X_train.shape[1]
model = build_model(input_dim)

# Define adaptive learning rate callback
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.9, patience=10, min_lr=1e-5, verbose=1)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=2000,
    batch_size=16,
    verbose=0,
    callbacks=[TqdmCallback(verbose=1), reduce_lr, early_stopping]
)

# Evaluate the model on the test set
y_pred = model.predict(X_test)
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("R-squared Score:", r2_score(y_test, y_pred))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



0epoch [00:00, ?epoch/s]

0batch [00:00, ?batch/s]


Epoch 28: ReduceLROnPlateau reducing learning rate to 0.0044999998994171625.

Epoch 71: ReduceLROnPlateau reducing learning rate to 0.004049999825656414.

Epoch 84: ReduceLROnPlateau reducing learning rate to 0.0036449996754527093.

Epoch 107: ReduceLROnPlateau reducing learning rate to 0.0032804996240884065.

Epoch 126: ReduceLROnPlateau reducing learning rate to 0.0029524497454985975.

Epoch 136: ReduceLROnPlateau reducing learning rate to 0.00265720474999398.

Epoch 146: ReduceLROnPlateau reducing learning rate to 0.002391484379768372.

Epoch 156: ReduceLROnPlateau reducing learning rate to 0.0021523358998820187.

Epoch 166: ReduceLROnPlateau reducing learning rate to 0.0019371022470295429.

Epoch 176: ReduceLROnPlateau reducing learning rate to 0.0017433920642361046.

Epoch 186: ReduceLROnPlateau reducing learning rate to 0.001569052878767252.

Epoch 196: ReduceLROnPlateau reducing learning rate to 0.0014121476328000427.

Epoch 206: ReduceLROnPlateau reducing learning rate to 0.00

In [208]:
# # Plot training and validation loss
# plt.plot(history.history['loss'], label='Training Loss')
# plt.plot(history.history['val_loss'], label='Validation Loss')
# plt.title('Model Loss')
# plt.xlabel('Epochs')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()

In [209]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=list(range(len(history.history['loss']))),
                         y=history.history['loss'],
                         mode='lines',
                         name='Training Loss'))

fig.add_trace(go.Scatter(x=list(range(len(history.history['val_loss']))),
                         y=history.history['val_loss'],
                         mode='lines',
                         name='Validation Loss'))

fig.update_layout(title='Model Loss',
                   xaxis_title='Epochs',
                   yaxis_title='Loss')

fig.show()

In [210]:
# Prediction for the given input
new_data = pd.DataFrame([
    ["audi", "hatchback", 99.5, 131, 160, 5500, 22]],
    columns=["make", "body-style", "wheel-base", "engine-size", "horsepower", "peak-rpm", "highway-mpg"]
)

new_data_processed = preprocessor.transform(new_data)
predicted_price = model.predict(new_data_processed)
print("Predicted Price:", predicted_price[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
Predicted Price: 19658.732


In [ ]:
# Pred 1 - Predicted Price: 20297.91